# 🧩 28 · Foundation Models for Microscopy
### One model, many microscopes: pretrain once, segment (almost) anything

**Part J · Research frontier** · Module 28 of 37 (frontier) · builds on 16, 19

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/28_foundation_models_for_microscopy.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (Colab free tier); a GPU speeds up the tiny U-Net but is not required |
| **Core frameworks** | NumPy, SciPy, scikit-image, PyTorch, Matplotlib |
| **Prerequisites** | Modules 16, 19 |
| **Estimated time** | 75–95 min |
| **Builds on** | Module 16 (deep learning classification) and Module 19 (deep-learning segmentation, U-Net) |
| **Research frontier covered** | Foundation models for microscopy — self-supervised pretraining, promptable/zero-shot segmentation (SAM, Cellpose/Cellpose-SAM, micro-SAM), and honest evaluation against a supervised specialist baseline |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every image in this notebook is SIMULATED by `simulate_nuclei_image` — there is no real microscopy data here, nothing is downloaded, and no pretrained foundation-model weights are loaded (they require a GPU and an internet connection this notebook does not assume you have).

## 🎯 Learning objectives

By the end of this module you will be able to:

1. Explain what a **foundation model** is in machine learning terms (pretrain once, at scale, on a broad
   objective; adapt or reuse the same weights for many downstream tasks) and why bioimage analysis moved
   toward this idea after a decade of training one segmentation network per dataset.
2. Describe, at the level of intuition and equations, two families of **self-supervised pretraining**
   objective (contrastive learning and masked-image modeling) and why they let a model learn useful
   structure from images that have no labels at all.
3. Explain the **promptable segmentation** idea popularised by the Segment Anything Model (SAM) — a click
   or box prompt plus a shared image embedding, decoded into a mask — and how microscopy-specific
   descendants (Cellpose, Cellpose-SAM, micro-SAM) adapt it to cells and nuclei.
4. Build and train a small supervised U-Net **specialist** segmenter, and compare it honestly against a
   training-free **generalist** pipeline on both an in-distribution test set and a held-out,
   out-of-distribution (OOD) shape mixture — and report, numerically, when each one wins.
5. Describe where current microscopy foundation models still fail, and what evidence you would need before
   trusting one in a real analysis pipeline.

## 🗺️ Contents

1. The biological question
2. Environment setup
3. What makes a model a *foundation* model?
4. Self-supervised pretraining, in one picture
5. Promptable, zero-shot segmentation: the SAM idea and its microscopy descendants
6. A fair fight: specialist U-Net vs. a training-free generalist pipeline
7. From toy to real data
8. Key takeaways, checkpoint, and further practice

## 🧫 The biological question

Every lab that images cells eventually asks the same question: *"how do I get accurate instance
segmentations of my nuclei/cells without annotating thousands of my own images by hand?"* For most of the
history of bioimage analysis the honest answer was "you don't — you annotate a training set for your own
microscope, cell line and stain, and train (or fine-tune) a network just for it," which is exactly what
Module 19 taught. That is expensive: every new imaging modality, magnification, or cell type restarts the
annotation effort.

The foundation-model idea asks whether one network, trained once on a very large and diverse collection of
images, can segment cells it has never specifically been trained on — new stains, new microscopes, new
organisms — well enough to use directly or to need only a light touch-up. This module gives you the
conceptual tools to understand what such a model actually is, why it can generalise at all, and — just as
importantly — how to **test** a generalist claim rather than take it on faith.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 20
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

Verbatim copies of the simulator and metric functions this notebook uses, identical to `course_toolkit.py`, so the notebook runs on its own.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def dice_coefficient(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Dice/F1 overlap of two binary masks: 2|A∩B| / (|A|+|B|)."""
    a, b = mask_a.astype(bool), mask_b.astype(bool)
    inter = (a & b).sum()
    denom = a.sum() + b.sum()
    return float(2 * inter / denom) if denom > 0 else 1.0


def iou_masks(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Intersection-over-union of two binary masks."""
    a, b = mask_a.astype(bool), mask_b.astype(bool)
    union = (a | b).sum()
    return float((a & b).sum() / union) if union > 0 else 1.0


def precision_recall_f1(y_true: np.ndarray, y_pred: np.ndarray):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 for binary labels
    (1 = positive). Returns (precision, recall, f1)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes


def watershed_labels(image: np.ndarray, min_distance: int = 7) -> np.ndarray:
    """Marker-controlled watershed segmentation of a blobby intensity image: finds markers at
    local intensity maxima (one per expected object) and floods from them. Returns an integer
    label image (0 = background)."""
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed as _watershed
    smooth = ndi.gaussian_filter(image, 1.0)
    coords = peak_local_max(smooth, min_distance=min_distance, labels=smooth > smooth.mean())
    markers = np.zeros(image.shape, dtype=int)
    for i, (y, x) in enumerate(coords, start=1):
        markers[y, x] = i
    markers = ndi.label(ndi.binary_dilation(markers > 0, iterations=2))[0] if coords.size else markers
    elevation = -smooth
    return _watershed(elevation, markers=markers, mask=smooth > smooth.mean() * 0.6)

---
## 1 · What makes a model a *foundation* model?

A **foundation model** is a network trained once, usually with a broad,
label-light objective, on a very large and diverse pile of data, whose weights are then *reused* — frozen,
lightly fine-tuned, or driven by a prompt — for many different downstream tasks, instead of training a new
network from scratch for each one.

Three properties distinguish this from the "one dataset, one model" workflow of Modules 16 and 19:

- **Scale of pretraining data.** Millions of images (often from many labs, modalities, and organisms) rather
  than one lab's few hundred annotated examples.
- **A broad, often self-supervised, pretraining objective** (Section 2) that does not require per-image
  human labels, so it can use data that would be far too expensive to annotate by hand.
- **Reuse without retraining from scratch.** The same pretrained weights adapt to a new task either with a
  **prompt** at inference time (Section 3) or with a small amount of fine-tuning — far less than training a
  specialist network would need.

*In words:* a specialist model answers one question extremely well because it has only ever seen that
question; a foundation model has seen so many related questions during pretraining that it already has
useful internal representations for a *new* question, even one it was never explicitly trained to answer.

### 📄 Research connection — why bioimage analysis needed this

For years, groups published a new segmentation network for every new cell type or imaging modality because
U-Net-style models (Module 19) generalise poorly outside the intensity statistics, cell density, and object
scale they were trained on. Re-annotating a training set for every new microscope or stain is slow and
expensive, and it does not scale to the sheer diversity of modern microscopy. The foundation-model direction
is a direct response: can we train (or adapt) *one* segmenter that works acceptably across many modalities
out of the box, saving most labs from ever training their own? Cellpose (Stringer et al. 2021) was an early,
hugely influential step in this direction for microscopy specifically, well before "foundation model"
became the standard term.

## 2 · Self-supervised pretraining, in one picture

Self-supervised pretraining manufactures a *supervision signal from the data
itself*, with no human labels. Two families dominate current foundation-model practice:

**Contrastive learning.** Take an image, make two randomly augmented views of it (crop, flip, colour jitter,
blur). Push the two views' embeddings *together* in representation space, and push embeddings of *different*
images *apart*. A common loss (InfoNCE) for a batch of $N$ paired views with embeddings $z_i, z_i^+$ is

$$\mathcal{L}_i = -\log \frac{\exp(\mathrm{sim}(z_i, z_i^+)/\tau)}{\sum_{j=1}^{N} \exp(\mathrm{sim}(z_i, z_j^+)/\tau)}$$

where $\mathrm{sim}(\cdot,\cdot)$ is cosine similarity and $\tau$ is a temperature. *In words:* the network
is never told "this is a nucleus" — it is only told "these two crops came from the same underlying image, so
represent them similarly; this other crop came from a different image, so represent it differently." Doing
this over millions of images forces the network to encode genuinely useful structure (edges, textures,
object-like blobs) because that is the easiest way to satisfy the contrastive objective.

**Masked-image modeling (MIM).** Hide random patches of the image and train the network to reconstruct
them (pixels, or a learned feature target) from the visible context, directly analogous to masked-language
modeling in text. *In words:* to fill in a missing patch of a plausible image, the network must have learned
what blobs, edges and textures normally look like — which is exactly the structure a downstream segmenter
needs.

Neither objective ever uses a segmentation label. That is the whole point: both can run on huge, unlabeled
image collections, which is what makes pretraining at foundation-model scale possible at all.

In [ ]:
# ── 2.1 A toy illustration: contrastive pretraining pulls augmented views together ──
# We will not train a real contrastive encoder (that needs millions of images); instead we SIMULATE
# what a successful contrastive objective achieves, to build the right intuition: augmented views of the
# SAME underlying image end up close together in embedding space, while different images end up far apart.

def toy_embedding(image, rng):
    """A deliberately crude 'embedding': a few coarse statistics of the image, standing in for what a
    trained encoder would compute. Good enough to illustrate the contrastive geometry, not a real encoder."""
    h, w = image.shape
    quadrants = [image[:h // 2, :w // 2], image[:h // 2, w // 2:], image[h // 2:, :w // 2], image[h // 2:, w // 2:]]
    return np.array([q.mean() for q in quadrants] + [image.std(), image.max()])

def augment(image, rng):
    """A random augmented 'view': small additive noise + a small random crop-and-resize stand-in
    (here, a roll) — enough to change pixels without changing the underlying content."""
    shift = rng.integers(-4, 5, size=2)
    view = np.roll(image, shift, axis=(0, 1))
    return add_gaussian_noise(view, 0.01, rng)

base_images = [simulate_nuclei_image(n_cells=10, size=80, seed=2000 + i)["image"] for i in range(5)]
view_pairs = [(augment(img, rng), augment(img, rng)) for img in base_images]

embeddings_1 = np.array([toy_embedding(v1, rng) for v1, v2 in view_pairs])
embeddings_2 = np.array([toy_embedding(v2, rng) for v1, v2 in view_pairs])
same_image_dist = np.linalg.norm(embeddings_1 - embeddings_2, axis=1)
cross_image_dist = np.array([np.linalg.norm(embeddings_1[i] - embeddings_2[j])
                              for i in range(5) for j in range(5) if i != j])
print(f"mean embedding distance, SAME image's two views : {same_image_dist.mean():.3f}")
print(f"mean embedding distance, DIFFERENT images        : {cross_image_dist.mean():.3f}")


🔍 **What just happened?** Even with our deliberately crude "embedding"
(a handful of quadrant means), two augmented views of the *same* simulated image land closer together in
embedding space than views of *different* images. A real contrastive encoder learns a far richer embedding
than four quadrant averages, but the geometric goal is identical: same-image views close, different-image
views far apart — and that geometry alone, learned from millions of unlabeled images, turns out to transfer
surprisingly well to downstream tasks like segmentation.

✏️ **Your turn.** Make `augment` noisier (larger `shift` range, bigger noise `sigma`). Does the gap between
"same image" and "different image" distances shrink? What does that tell you about how much distortion a
contrastive objective can tolerate before it stops being a useful training signal?

## 3 · Promptable, zero-shot segmentation: the SAM idea and its microscopy descendants

The Segment Anything Model (SAM, Kirillov et al. 2023) popularised a specific
architecture for turning a foundation-scale pretrained encoder into a segmentation tool without retraining
it per task:

1. A large image encoder (pretrained, usually frozen) turns the whole image into a rich embedding, computed
   **once per image**.
2. A lightweight **prompt encoder** turns a user's click point, box, or rough mask into a prompt embedding.
3. A small, fast **mask decoder** combines the image embedding and the prompt embedding to output a mask —
   in milliseconds, because only the tiny decoder has to run per prompt; the expensive encoder pass is
   shared across every prompt on that image.

*In words:* instead of training a whole new network for "segment nuclei" versus "segment mitochondria," you
train one powerful general-purpose image encoder plus a tiny, cheap "turn this hint into a mask" head, and
you supply the task-specific knowledge as a *prompt* at inference time rather than as network weights.

**Microscopy-specific descendants.** SAM itself was trained on natural photographs and segments microscopy
images only moderately well out of the box — cells and nuclei have very different size scales, contrast
mechanisms and crowding patterns than everyday photos. Two influential adaptations:

- **Cellpose** (Stringer et al. 2021) predates SAM but solves a closely related generalisation problem for
  microscopy specifically: instead of predicting a mask directly, it predicts a vector flow field that
  points every pixel toward its object's centre, then finds instances by following the flows to their
  sinks — a trick that handles touching, variably-sized cells without knowing the object's diameter in
  advance (hence *diameter-agnostic*). Cellpose 2.0 and the SAM-influenced *Cellpose-SAM* line extended the
  model's training diversity so that a single released model generalises across many microscopy modalities.
- **micro-SAM** (Archit et al. 2025) instead fine-tunes SAM's image encoder on a large, diverse collection of
  microscopy images while keeping the prompt-and-decode interaction, bringing SAM's promptable, zero-shot
  workflow natively into light and electron microscopy.

Both directions chase the same goal from different starting points: Cellpose generalised a microscopy-native
architecture across modalities; micro-SAM adapted a natural-image foundation model into microscopy.

### 📄 Research connection — why one model can span so many microscopes

Despite wildly different optics and stains, most microscopy images that contain discrete objects (nuclei,
cells, vesicles, droplets) share low-level statistics: blobby or elongated bright regions on a smoother
background, with object boundaries marked by local intensity discontinuities. A model pretrained across many
such datasets can learn these shared regularities once, rather than re-learning "a blob is a blob" from
scratch for every new dataset. This is also exactly where generalist models still fail: imaging conditions
that violate the shared assumption — extremely dense, touching objects; unusual markers that don't look
blob-like; very low signal-to-noise regimes — are where a specialist, trained specifically on that data,
still tends to win. Section 4 makes this trade-off numerically concrete.

## 4 · A fair fight: specialist U-Net vs. a training-free generalist pipeline

We cannot download and run a real pretrained foundation model here — SAM,
Cellpose-SAM and micro-SAM all ship multi-hundred-megabyte pretrained weights and are normally run on a GPU
with an internet connection, neither of which this self-contained, data-free notebook assumes. Instead we
build a **training-free classical pipeline** — threshold + marker-controlled watershed, using only pixel
intensities, with *no* learned parameters and *no* per-dataset training required — as a stand-in for the one
property that matters most for this comparison: a method that works on a new image distribution **without
retraining**. Real foundation models are far more capable than this classical stand-in on genuinely novel
imagery (that is the whole point of pretraining at scale), but the retrain-free property is exactly
analogous, and it lets us run a scientifically honest, offline, same-notebook comparison.

Against it we train a genuine specialist: a small U-Net (same idea as Module 19), trained **only** on
round-nucleus images, and ask: which one holds up better when the test images stop looking like the
training images?

In [ ]:
# ── 4.1 Three data regimes: a training set, and two held-out test sets ──
N_TRAIN, N_TEST = 36, 10
IMG_SIZE = 80

def make_split(n, shape_classes, seed0):
    imgs, masks = [], []
    for i in range(n):
        sim = simulate_nuclei_image(n_cells=10, size=IMG_SIZE, shape_classes=shape_classes, seed=seed0 + i)
        imgs.append(sim["image"].astype(np.float32))
        masks.append((sim["instance_mask"] > 0).astype(np.float32))
    return imgs, masks

train_imgs, train_masks = make_split(N_TRAIN, ("round",), seed0=3000)
test_id_imgs, test_id_masks = make_split(N_TEST, ("round",), seed0=4000)
test_ood_imgs, test_ood_masks = make_split(N_TEST, ("kidney", "ring", "dividing", "irregular"), seed0=5000)

show_image_grid(train_imgs[:3] + test_ood_imgs[:3],
                 titles=["train (round)"] * 3 + ["OOD test (mixed)"] * 3, ncols=3)
plt.show()


🔍 **What just happened?** The *training* distribution and the *in-distribution (ID) test* distribution both contain only round nuclei. The *out-of-distribution (OOD) test* set swaps in kidney-shaped, ring-shaped, dividing and irregular shapes the U-Net never saw during training — a controlled, honest stand-in for 'a new lab sends you images from a different cell line.'

In [ ]:
# ── 4.2 A tiny U-Net specialist (same spirit as Module 19, deliberately small) ──
import torch
import torch.nn as nn

class TinyUNet(nn.Module):
    def __init__(self, base=8):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, base, 3, padding=1), nn.ReLU(),
                                   nn.Conv2d(base, base, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(base, base * 2, 3, padding=1), nn.ReLU(),
                                   nn.Conv2d(base * 2, base * 2, 3, padding=1), nn.ReLU())
        self.up = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(base * 2, base, 3, padding=1), nn.ReLU(),
                                   nn.Conv2d(base, base, 3, padding=1), nn.ReLU())
        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        d1 = self.dec1(torch.cat([self.up(e2), e1], dim=1))
        return self.head(d1)

torch.manual_seed(SEED)
unet = TinyUNet()
opt = torch.optim.Adam(unet.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

X_train = torch.tensor(np.stack(train_imgs))[:, None, :, :].float()
Y_train = torch.tensor(np.stack(train_masks))[:, None, :, :].float()

train_losses = []
for epoch in range(25):
    opt.zero_grad()
    pred = unet(X_train)
    loss = loss_fn(pred, Y_train)
    loss.backward()
    opt.step()
    train_losses.append(loss.item())

plt.figure(figsize=(4, 3))
plt.plot(train_losses)
plt.xlabel("epoch"); plt.ylabel("training BCE loss"); plt.title("TinyUNet specialist — training curve")
plt.tight_layout(); plt.show()
print(f"final training loss: {train_losses[-1]:.4f}")


🔍 **What just happened?** The specialist U-Net trained for 25 quick epochs on CPU, using only 36 round-nucleus images — tiny by deep-learning standards, but enough to memorise this one narrow distribution well, which is exactly the scenario where a specialist should shine.

In [ ]:
# ── 4.3 The training-free generalist stand-in: threshold + watershed, no learned weights ──
def generalist_segment(image):
    labels = watershed_labels(image, min_distance=6)
    return (labels > 0).astype(np.float32)

def unet_segment(image):
    with torch.no_grad():
        x = torch.tensor(image[None, None, :, :]).float()
        prob = torch.sigmoid(unet(x))[0, 0].numpy()
    return (prob > 0.5).astype(np.float32)

def evaluate(method, imgs, masks):
    dices = [dice_coefficient(method(img), m) for img, m in zip(imgs, masks)]
    ious = [iou_masks(method(img), m) for img, m in zip(imgs, masks)]
    return np.mean(dices), np.mean(ious)

results = {}
for split_name, imgs, masks in [("in-distribution (round)", test_id_imgs, test_id_masks),
                                 ("out-of-distribution (mixed)", test_ood_imgs, test_ood_masks)]:
    for method_name, method in [("specialist U-Net", unet_segment), ("generalist (no training)", generalist_segment)]:
        dice, iou = evaluate(method, imgs, masks)
        results[(split_name, method_name)] = (dice, iou)
        print(f"{split_name:30s} | {method_name:26s} | Dice {dice:.3f} | IoU {iou:.3f}")


🧪 **Ground-truth check.** Because every mask comes straight from the simulator
that drew the image, Dice and IoU here are exact, noise-free measurements against a known truth — not an
estimate from imperfect human annotation. Look at the printed numbers: does the specialist U-Net's score
drop more, less, or about the same as the generalist's when you move from the in-distribution test set to
the out-of-distribution one?

In [ ]:
# ── 4.4 Visualise one in-distribution and one OOD example side by side ──
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for row, (img, mask, label) in enumerate([
        (test_id_imgs[0], test_id_masks[0], "in-distribution"),
        (test_ood_imgs[0], test_ood_masks[0], "out-of-distribution")]):
    show_image(axes[row, 0], img, f"{label}: input")
    show_image(axes[row, 1], unet_segment(img), "specialist U-Net")
    show_image(axes[row, 2], generalist_segment(img), "generalist (watershed)")
fig.tight_layout(); plt.show()


✏️ **Your turn.** Retrain `unet` (re-run cell 4.2) after widening the training
distribution to `shape_classes=("round", "kidney")` instead of just `("round",)`. Re-run the evaluation in
4.3 using the *same* OOD test set (which still includes ring/dividing/irregular shapes the retrained model
has never seen). Does a small amount of added training diversity shrink the gap to the generalist — the
same effect that motivates training foundation models on far larger and more diverse data than any single
lab could collect?

**Reading the result honestly.** A specialist trained tightly on one narrow
distribution will typically win decisively in-distribution (it has memorised exactly the shapes, size range
and noise level of the test set) and lose more of its edge, sometimes dramatically, out-of-distribution. A
training-free generalist pipeline usually trails the specialist in-distribution but degrades far more
gracefully out-of-distribution, because it never relied on the training shapes in the first place — it is
reading pixel contrast, not matching a memorised appearance. This is the honest version of the "foundation
models generalise better" claim: it is not that a foundation model is simply *better* in some absolute
sense, it is that its performance is far less *sensitive to distribution shift*, at some up-front cost in
peak in-distribution accuracy. A rigorous evaluation always reports both regimes, never just one.

## 5 · From toy to real data

To move from this notebook's classical stand-in to the real pretrained
foundation models it represents:

- **Cellpose / Cellpose-SAM** (`pip install cellpose`; Stringer et al. 2021, *Nature Methods*; Pachitariu &
  Stringer 2022, *Nature Methods*) ships ready-to-run pretrained generalist models for nucleus and
  cytoplasm segmentation across many microscopy modalities, usable from Python or a GUI, with optional
  fine-tuning on your own small annotated set.
- **micro-SAM** (Archit et al. 2025) adapts Meta AI's Segment Anything Model (Kirillov et al. 2023,
  *ICCV*) to light and electron microscopy, with an interactive napari plugin for point/box-prompted
  segmentation and correction.
- The **BioImage Model Zoo** (bioimage.io) hosts dozens of pretrained bioimage models, including several of
  the above, in a shared format runnable from Fiji, ilastik, napari, or directly in Python — the practical
  place to go looking for "has someone already solved this" before training anything yourself.

A realistic next step with real data: segment one of your own fluorescence images with Cellpose's
out-of-the-box generalist model, then compare its output, by eye and with Dice against a small hand-drawn
ground truth, to a U-Net you fine-tune on just 10–20 of your own annotated images — the exact experiment
this module ran in miniature.

## 📌 Key takeaways

- A foundation model is pretrained once, broadly and often self-supervised, then reused — by prompting or
  light fine-tuning — across many downstream tasks, instead of being trained from scratch per task.
- Contrastive learning and masked-image modeling are the two dominant self-supervised objectives; neither
  needs human segmentation labels, which is what makes pretraining at huge scale feasible.
- SAM's promptable architecture (shared image encoder + cheap prompt-driven mask decoder) was adapted to
  microscopy by Cellpose/Cellpose-SAM and micro-SAM, each starting from a different side of the problem.
- A specialist model trained narrowly tends to win in-distribution and lose more out-of-distribution; a
  generalist/training-free method trades some peak accuracy for much steadier behaviour under distribution
  shift — always evaluate both regimes before trusting a generalisation claim.

## ✅ Checkpoint

1. What is the key difference between how a specialist segmentation network and a promptable foundation
   model are adapted to a new dataset?
   <details><summary>Answer</summary>A specialist is retrained (or fine-tuned) on new labeled data from that
   dataset. A promptable foundation model is adapted at inference time via a prompt (click, box) using
   frozen or lightly-adapted weights, with no per-dataset retraining required.</details>

2. Why does masked-image modeling not need segmentation labels to be useful for later segmentation tasks?
   <details><summary>Answer</summary>Its training signal comes from predicting hidden patches of the image
   itself, which forces the network to learn the structure of blobs, edges and textures — the same features
   a downstream segmenter needs — without ever being told where any object's boundary is.</details>

3. In Section 4, which method's Dice score dropped more between the in-distribution and out-of-distribution
   test sets, and why does that match what the theory in Section 3 predicts?
   <details><summary>Answer</summary>The specialist U-Net typically drops more, because it was trained on
   only the round-shape distribution and has no mechanism for handling shapes it never saw; the
   training-free watershed pipeline uses the same pixel-intensity logic regardless of shape, so it degrades
   less.</details>

4. Name one property of a real pretrained foundation model (e.g. Cellpose-SAM) that this notebook's
   watershed stand-in does *not* actually share.
   <details><summary>Answer</summary>Any reasonable answer: the stand-in has no learned representation at
   all, cannot be prompted with a click/box, was not trained on millions of diverse images, and does not
   improve with fine-tuning the way a pretrained network's weights would.</details>

## 🧠 Questions & discussion

1. If a foundation model's pretraining data under-represents a particular imaging modality (say,
   label-free phase-contrast time-lapses), should you expect its zero-shot performance there to resemble its
   performance on well-represented modalities? What would you want to measure before trusting it?
2. Cellpose's flow-field representation lets it handle touching, variable-diameter cells without knowing
   object size in advance. What specifically about predicting a *direction* at every pixel, rather than a
   binary mask directly, makes that possible?
3. A colleague claims "our new foundation model beats every specialist on every benchmark." What experimental
   design from Section 4 would you ask them to show before accepting that claim?

## 🚀 Bonus — going further

1. Add a third, intermediate test set — say `shape_classes=("round", "elongated")` — and re-run the
   evaluation. Does performance degrade smoothly with "distance" from the training distribution, or is
   there a sharper cliff?
2. Replace the watershed generalist with a model trained on *all six* shape classes at once (still a
   specialist by our definition, since it still needed retraining — but now trained more broadly). How does
   its OOD performance compare to both the narrow specialist and the training-free generalist?
3. Read the Cellpose paper's description of the flow-field representation and sketch, in your own words
   (no code required), how you would decode a predicted flow field back into separate object instances.

## 📝 Lab note

```text
Module 28 — Foundation models for microscopy
Date:
Training distribution used for the specialist U-Net:
In-distribution Dice / IoU (specialist vs. generalist):
Out-of-distribution Dice / IoU (specialist vs. generalist):
Biggest surprise in the comparison:
One real foundation model I would try next on my own data, and why:
```

## 📚 References

1. Stringer, C., Wang, T., Michaelos, M. & Pachitariu, M. (2021). Cellpose: a generalist algorithm for
   cellular segmentation. *Nature Methods*, 18, 100–106.
2. Pachitariu, M. & Stringer, C. (2022). Cellpose 2.0: how to train your own model. *Nature Methods*, 19,
   1634–1641.
3. Kirillov, A. et al. (2023). Segment Anything. *Proceedings of the IEEE/CVF International Conference on
   Computer Vision (ICCV)*.
4. Archit, A. et al. (2025). Segment Anything for Microscopy (micro-SAM). *Nature Methods*.
5. Chen, T., Kornblith, S., Norouzi, M. & Hinton, G. (2020). A Simple Framework for Contrastive Learning of
   Visual Representations (SimCLR). *ICML*.
6. He, K., Chen, X., Xie, S., Li, Y., Dollár, P. & Girshick, R. (2022). Masked Autoencoders Are Scalable
   Vision Learners. *CVPR*.
7. Ouyang, W. et al. (2019). BioImage Model Zoo / BioImage.IO: a community-driven resource for sharing
   deep-learning models in bioimage analysis.
8. Ronneberger, O., Fischer, P. & Brox, T. (2015). U-Net: Convolutional Networks for Biomedical Image
   Segmentation. *MICCAI*.

## ➡️ Next

Module 29 stays in the deep-learning frontier but turns from *segmenting* images to *generating* them:
virtual staining, where a network predicts a fluorescent or histological stain from an unstained image —
and where the hallucination risk of a confident-but-wrong prediction is even more consequential.